# Qwen2.5-VL-7B QLoRA 파인튜닝 v2

v1 대비 변경사항:
- BBQ 전체 데이터 사용 (train 52,643 + val 5,849 = 58,492개)
- LoRA target modules에 FFN 레이어 추가 (gate_proj, up_proj, down_proj)
- LoRA rank 16 → 32
- 학습 완료 후 HuggingFace Hub에 자동 업로드

**실행 전 확인사항**
- Accelerator: **GPU T4 x2 or A100**
- Internet: **ON**
- Add data: `skku-bbq-data` 데이터셋 추가
- Secrets: `HF_TOKEN` 등록

## 1. 패키지 설치

In [ ]:
!pip install -q \
    transformers>=4.49.0 \
    peft>=0.14.0 \
    bitsandbytes>=0.43.0 \
    trl>=0.12.0 \
    accelerate>=0.26.0 \
    qwen-vl-utils

## 2. 라이브러리 임포트

In [ ]:
import json
import os
from pathlib import Path

import torch
import pandas as pd
from torch.utils.data import Dataset
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from kaggle_secrets import UserSecretsClient

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. 설정

In [ ]:
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["HF_HUB_DISABLE_XET"] = "1"

# ── 경로 ─────────────────────────────────────────────────────
BBQ_DATA_DIR = "/kaggle/input/datasets/teddykwj/skku-bbq-data/bbq_data"
TRAIN_CSV    = f"{BBQ_DATA_DIR}/train.csv"
VAL_CSV      = f"{BBQ_DATA_DIR}/val.csv"
OUTPUT_DIR   = "/kaggle/working/qwen_qlora_v2"

# ── 모델 ──────────────────────────────────────────────────────
MODEL_ID       = "Qwen/Qwen2.5-VL-7B-Instruct"
ADAPTER_HUB_ID = "teddykwj/qwen-bbq-lora-v2"

# ── LoRA (v2: FFN 추가, rank 32) ──────────────────────────────
LORA_RANK      = 32
LORA_ALPHA     = 64
LORA_DROPOUT   = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # Attention
    "gate_proj", "up_proj", "down_proj",        # FFN
]

# ── 학습 ──────────────────────────────────────────────────────
MAX_SEQ_LEN       = 768
MAX_TRAIN_SAMPLES = 40000
BATCH_SIZE        = 1
GRAD_ACCUM        = 16
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 1
WARMUP_RATIO      = 0.05
SEED              = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
assert Path(TRAIN_CSV).exists(), f"❌ train.csv 없음: {TRAIN_CSV}"
assert Path(VAL_CSV).exists(),   f"❌ val.csv 없음: {VAL_CSV}"

# train + val 합산 후 40,000개 샘플링 (ambig/disambig 50:50 유지)
full_df = pd.concat([
    pd.read_csv(TRAIN_CSV),
    pd.read_csv(VAL_CSV),
]).reset_index(drop=True)

half     = MAX_TRAIN_SAMPLES // 2
ambig    = full_df[full_df["context_condition"] == "ambig"].sample(n=min(half, (full_df["context_condition"]=="ambig").sum()), random_state=SEED)
disambig = full_df[full_df["context_condition"] == "disambig"].sample(n=min(half, (full_df["context_condition"]=="disambig").sum()), random_state=SEED)
full_df  = pd.concat([ambig, disambig]).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"학습 샘플: {len(full_df):,}개")
print(f"  ambig    : {(full_df['context_condition']=='ambig').sum():,}")
print(f"  disambig : {(full_df['context_condition']=='disambig').sum():,}")

## 4. 모델 로드 (4-bit QLoRA)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    token=os.environ["HF_TOKEN"],
)
model.config.use_cache = False

processor = AutoProcessor.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])

print(f"모델 로드 완료")
print(f"사용 VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 5. LoRA 적용 (v2: FFN 추가, rank 32)

In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. 데이터셋

In [ ]:
class BBQDataset(Dataset):
    def __init__(self, df, processor, max_length=768):
        self.df        = df.reset_index(drop=True)
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def _build_reason(self, row):
        if row.get("context_condition", "") == "disambig":
            return "The context provides explicit evidence to determine the answer."
        return "The context does not provide sufficient information. Cannot determine without stereotyping."

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        answers = json.loads(row["answers"])
        label   = int(row["label"])

        user_text = (
            "You are a fair and unbiased assistant. "
            "Answer ONLY based on explicit information in the context. "
            "If there is no explicit evidence, choose the 'cannot be determined' option.\n\n"
            f"Context: {row['context']}\n"
            f"Question: {row['question']}\n"
            f"Options:\n0. {answers[0]}\n1. {answers[1]}\n2. {answers[2]}\n\n"
            '{"reason": "...", "answer_id": "0"|"1"|"2"}'
        )
        assistant_text = json.dumps({
            "reason":    self._build_reason(row),
            "answer_id": str(label),
        })

        messages = [
            {"role": "user",      "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ]

        full_text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        user_only = self.processor.apply_chat_template(
            [{"role": "user", "content": user_text}],
            tokenize=False, add_generation_prompt=True
        )

        full_enc = self.processor(
            text=[full_text], return_tensors="pt",
            max_length=self.max_length, truncation=True,
        )
        user_enc = self.processor(
            text=[user_only], return_tensors="pt", truncation=True,
        )

        input_ids      = full_enc["input_ids"][0]
        attention_mask = full_enc["attention_mask"][0]
        labels         = input_ids.clone()
        labels[:user_enc["input_ids"].shape[1]] = -100

        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


train_dataset = BBQDataset(full_df, processor, MAX_SEQ_LEN)

print(f"Train: {len(train_dataset):,}")
sample = train_dataset[0]
print(f"input_ids shape : {sample['input_ids'].shape}")
print(f"학습 토큰 수     : {(sample['labels'] != -100).sum().item()}")
print("✅ 데이터셋 OK")

## 7. 데이터 콜레이터

In [ ]:
def collate_fn(batch):
    pad_id = processor.tokenizer.pad_token_id or 0

    input_ids = torch.nn.utils.rnn.pad_sequence(
        [b["input_ids"] for b in batch], batch_first=True, padding_value=pad_id
    )
    attention_mask = torch.nn.utils.rnn.pad_sequence(
        [b["attention_mask"] for b in batch], batch_first=True, padding_value=0
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        [b["labels"] for b in batch], batch_first=True, padding_value=-100
    )

    return {
        "input_ids":      input_ids,
        "attention_mask": attention_mask,
        "labels":         labels,
    }

## 8. 학습

In [ ]:
import time
from transformers import TrainerCallback

class LogCallback(TrainerCallback):
    def __init__(self):
        self.start_time = time.time()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or state.global_step == 0:
            return
        elapsed = (time.time() - self.start_time) / 3600
        loss = logs.get("loss", "N/A")
        lr   = logs.get("learning_rate", "N/A")
        pct  = state.global_step / state.max_steps * 100 if state.max_steps else 0
        if isinstance(loss, float):
            print(
                f"[{elapsed:.1f}h] Step {state.global_step}/{state.max_steps} "
                f"({pct:.1f}%)  loss={loss:.4f}  lr={lr:.2e}",
                flush=True,
            )

class TimeoutCallback(TrainerCallback):
    def __init__(self, max_hours=11.0):
        self.deadline = time.time() + max_hours * 3600

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() > self.deadline:
            print("⚠️  11시간 경과 — 학습 조기 종료 후 모델 저장 진행", flush=True)
            control.should_training_stop = True
        return control


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=50,
    eval_strategy="no",
    save_strategy="no",
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
    callbacks=[LogCallback(), TimeoutCallback(max_hours=11.0)],
)

trainer.train()

## 9. 로컬 저장 및 HuggingFace Hub 업로드

In [ ]:
adapter_dir = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(adapter_dir)
processor.save_pretrained(adapter_dir)

print(f"✅ 로컬 저장 완료: {adapter_dir}")
for f in sorted(Path(adapter_dir).iterdir()):
    print(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")

print(f"\nHuggingFace Hub 업로드 중: {ADAPTER_HUB_ID}")
model.push_to_hub(ADAPTER_HUB_ID, token=os.environ["HF_TOKEN"])
processor.push_to_hub(ADAPTER_HUB_ID, token=os.environ["HF_TOKEN"])
print(f"✅ 업로드 완료: https://huggingface.co/{ADAPTER_HUB_ID}")